# Colab Inference Server

Serves a single HTTP endpoint over an ngrok tunnel for the campus
navigation app:
- `POST /analyze` -- image extraction with Gemma 4 E4B-it (vision-language).

Model loading, system prompt and decoding settings are copied verbatim from
`extraction_validation/extraction_validation.ipynb` so live inference matches
the validated accuracy metrics (no template drift between offline and runtime).

STT (Whisper q5 tiny) runs on-device inside the Flutter app, so this
notebook does not need to host a `/transcribe` endpoint.

**Run order:** execute cells top to bottom. Cell 1 restarts the runtime so the
pinned Pillow loads cleanly.


In [ ]:
# Install dependencies. Pillow is pinned to avoid a known Colab runtime
# conflict with the _Ink module. The runtime is killed at the end so the
# new Pillow version loads cleanly on restart.
!pip install -q "Pillow==10.4.0"
!pip install -q -U \
    "transformers>=4.47.0" \
    torchvision \
    accelerate \
    sentencepiece \
    fastapi \
    uvicorn \
    python-multipart \
    pyngrok \
    huggingface_hub \
    nest_asyncio

import os
os.kill(os.getpid(), 9)


## 1. Credentials


In [ ]:
import os
from google.colab import userdata

# Load secrets from Colab's secret manager (key icon in the left sidebar).
# Required secrets: HF_TOKEN, NGROK_TOKEN.
hf_token    = userdata.get("HF_TOKEN")
ngrok_token = userdata.get("NGROK_TOKEN")

if not hf_token:    raise ValueError("Missing HF_TOKEN. Add it in Colab Secrets.")
if not ngrok_token: raise ValueError("Missing NGROK_TOKEN. Add it in Colab Secrets.")

os.environ["HF_TOKEN"]         = hf_token
os.environ["NGROK_AUTH_TOKEN"] = ngrok_token

print("Secrets loaded.")


## 2. Hugging Face Login

Gemma 4 E4B-it is a gated model. The HF token must have access granted for the
`google/gemma-4-E4B-it` repository.


In [ ]:
from huggingface_hub import login, whoami

login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)
print("Logged in as:", whoami()["name"])


## 3. Inference Server

The server module is written to disk so `uvicorn` can import it as a normal
Python module on port 8000.


In [ ]:
%%writefile server.py
"""HTTP server for the campus navigation Colab runtime.

Single endpoint:
- POST /analyze -> image + prompt processed by Gemma 4 E4B-it.

The inference path mirrors the validation notebook exactly: deterministic
decoding (do_sample=False, num_beams=1, seed=0), the same SYSTEM_PROMPT,
and the same chat-template messages structure. The Windows backend
(src/api/services/image_analysis.py) builds prompts from the validated
catalogue (extraction_validation/prompts.json) and POSTs them here, so
production output is reproducible against the offline metrics.

STT (Whisper q5 tiny) runs on-device inside the Flutter app; the legacy
/transcribe endpoint is no longer needed.
"""
import io
import os
import random

import numpy as np
import torch
from fastapi import FastAPI, File, Form, UploadFile
from fastapi.middleware.cors import CORSMiddleware
from PIL import Image
from transformers import AutoModelForCausalLM, AutoProcessor

# Reproducibility (matches extraction_validation.ipynb).
SEED = 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Credentials.
HF_TOKEN = os.environ["HF_TOKEN"]

# Model loading.
MODEL_ID = "google/gemma-4-E4B-it"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Loading {MODEL_ID} on {DEVICE} ...")
processor = AutoProcessor.from_pretrained(MODEL_ID, token=HF_TOKEN)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN,
    dtype="auto" if DEVICE == "cuda" else torch.float32,
    device_map="auto" if DEVICE == "cuda" else None,
).eval()

n_params_b = sum(p.numel() for p in model.parameters()) / 1e9
print(f"Loaded. dtype={model.dtype} | params={n_params_b:.2f}B")
if DEVICE == "cuda":
    print(
        f"GPU: {torch.cuda.get_device_name(0)} | "
        f"VRAM allocated: {torch.cuda.memory_allocated() / 1024 ** 3:.2f} GB"
    )

# Inference contracts copied verbatim from the validation notebook.
SYSTEM_PROMPT = (
    "You are a strict format extractor for indoor navigation. "
    "Read the question, follow its rules exactly, and output only what the "
    "question asks. Never invent identifiers. Never add explanations or extra "
    "words. Stop generating immediately after the requested output is complete."
)
DEFAULT_MAX_NEW_TOKENS = 32

app = FastAPI(title="Campus Navigation Inference Server")
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)


@app.get("/")
async def root():
    return {"status": "running", "model": MODEL_ID, "device": DEVICE}


def _strip_legacy_image_marker(prompt: str) -> str:
    """Drop a leading ``<image>\n`` left over from the PaliGemma era.

    Gemma 4 attaches the image through the chat-template message, so a
    literal ``<image>`` token in the user text would be sent as plain
    characters and degrade the response. The current Windows backend no
    longer prepends it, but this strips defensively in case an older
    client calls this endpoint.
    """
    if prompt.startswith("<image>"):
        _, _, rest = prompt.partition("\n")
        return rest
    return prompt


@app.post("/analyze")
async def analyze_image(
    file: UploadFile = File(...),
    prompt: str = Form(...),
    max_tokens: str = Form(None),
):
    """Run one validated prompt against an image. Returns {success, response, error}."""
    try:
        image_bytes = await file.read()
        image = Image.open(io.BytesIO(image_bytes)).convert("RGB")

        max_new_tokens = (
            int(max_tokens)
            if (max_tokens and max_tokens.isdigit())
            else DEFAULT_MAX_NEW_TOKENS
        )
        prompt_text = _strip_legacy_image_marker(prompt)

        messages = [
            {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
            {"role": "user", "content": [
                {"type": "image", "image": image},
                {"type": "text",  "text": prompt_text},
            ]},
        ]

        inputs = processor.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
            enable_thinking=False,
        ).to(model.device)
        inputs = inputs.to(dtype=model.dtype)

        with torch.no_grad():
            output = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                num_beams=1,
                pad_token_id=(
                    processor.tokenizer.pad_token_id
                    or processor.tokenizer.eos_token_id
                ),
                eos_token_id=processor.tokenizer.eos_token_id,
            )

        input_len = inputs["input_ids"].shape[-1]
        response = processor.decode(
            output[0][input_len:], skip_special_tokens=True
        ).strip()

        print(f"[/analyze] prompt={prompt_text[:80]!r} -> {response!r}")
        return {"success": True, "response": response, "error": None}

    except Exception as exc:
        print(f"[/analyze] ERROR: {exc}")
        return {"success": False, "response": "", "error": str(exc)}


## 4. ngrok Tunnel


In [ ]:
import time

from pyngrok import ngrok, conf

PORT = 8000

conf.get_default().auth_token = os.environ["NGROK_AUTH_TOKEN"]

# Close any leftover tunnels before opening a new one.
try:
    for tunnel in ngrok.get_tunnels():
        ngrok.disconnect(tunnel.public_url)
except Exception:
    pass
ngrok.kill()
time.sleep(2)

tunnel = ngrok.connect(PORT)
public_url = tunnel.public_url

print("=" * 55)
print(f"  Public URL: {public_url}")
print("\n  Set this in your Windows .env file:")
print(f"  COLAB_API_URL={public_url}")
print("=" * 55)
print("\n  Run the next cell to start the server.")


## 5. Start Server

This cell runs indefinitely while the server is active.
To stop it: **Runtime > Interrupt execution**.


In [ ]:
import nest_asyncio
import uvicorn

# nest_asyncio lets uvicorn run inside Colab's existing event loop.
nest_asyncio.apply()

print("Starting server...")
print("Gemma 4 E4B-it takes a few minutes to load on first run.")
print("Wait for 'Application startup complete.' before using the app.\n")

config = uvicorn.Config("server:app", host="0.0.0.0", port=8000, log_level="info")
server = uvicorn.Server(config)
await server.serve()


In [ ]:
# Run this cell to verify the tunnel is reachable.
# Only execute after seeing "Application startup complete." in the cell above.
import requests

try:
    response = requests.get(f"{public_url}/", timeout=5)
    print("Server ready:", response.json())
except Exception as exc:
    print(f"Not ready yet, wait a moment and try again: {exc}")
